In [ ]:
!pip install duckdb -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import duckdb
import os

In [ ]:
file_path = "/content/drive/MyDrive/RDV_Project/processed_data/taxi_weather.parquet"

df = pd.read_parquet(file_path)

print(df.shape)
df.head()

(22012724, 16)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,month,trip_duration,pickup_hour,time_category,trip_category,pickup_date,avg_temperature,precipitation,rain,weather_condition
0,2025-01-01 00:18:38,2025-01-01 00:26:59,229,237,1.60,10.0,2025-01,8.350000,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
1,2025-01-01 00:32:40,2025-01-01 00:35:13,236,237,0.50,5.1,2025-01,2.550000,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
2,2025-01-01 00:44:04,2025-01-01 00:46:01,141,141,0.60,5.1,2025-01,1.950000,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
3,2025-01-01 00:14:27,2025-01-01 00:20:01,244,244,0.52,7.2,2025-01,5.566667,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan
4,2025-01-01 00:21:34,2025-01-01 00:25:06,244,116,0.66,5.8,2025-01,3.533333,0,Malam,Pendek,2025-01-01,7.4,4.5,4.5,Hujan Ringan


In [ ]:
db_path = "/content/drive/MyDrive/RDV_Project/processed_data/rdv_project.duckdb"

con = duckdb.connect(db_path)

print("Connected")

Connected


In [ ]:
dim_time = df[
    [
        "pickup_date",
        "pickup_hour",
        "time_category"
    ]
].drop_duplicates()

dim_time = dim_time.reset_index(
    drop=True
)

dim_time["time_id"] = (
    dim_time.index + 1
)

dim_time.head()

,pickup_date,pickup_hour,time_category,time_id
0,2025-01-01,0,Malam,1
1,2025-01-01,1,Malam,2
2,2024-12-31,23,Malam,3
3,2024-12-31,20,Malam,4
4,2024-12-31,21,Malam,5


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_time AS
SELECT * FROM dim_time
""")

In [ ]:
pickup_loc = df[
    ["PULocationID"]
].rename(
    columns={
        "PULocationID":
        "location_id"
    }
)

dropoff_loc = df[
    ["DOLocationID"]
].rename(
    columns={
        "DOLocationID":
        "location_id"
    }
)

dim_location = pd.concat(
    [
        pickup_loc,
        dropoff_loc
    ]
).drop_duplicates()

dim_location = (
    dim_location
    .reset_index(drop=True)
)

dim_location.head()

,location_id
0,229
1,236
2,141
3,244
4,239


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_location AS
SELECT * FROM dim_location
""")

In [ ]:
fact_weather = df[
    [
        "pickup_date",
        "avg_temperature",
        "precipitation",
        "rain",
        "weather_condition"
    ]
].drop_duplicates()

fact_weather.head()

,pickup_date,avg_temperature,precipitation,rain,weather_condition
0,2025-01-01,7.4,4.5,4.5,Hujan Ringan
592,2024-12-31,NaN,NaN,NaN,None
70826,2025-01-02,2.6,0.0,0.0,Tidak Hujan
146767,2025-01-03,0.4,0.0,0.0,Tidak Hujan
229921,2025-01-04,-1.4,0.0,0.0,Tidak Hujan


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE fact_weather AS
SELECT * FROM fact_weather
""")

In [ ]:
fact_taxi_trip = df[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pickup_date",
        "pickup_hour",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "trip_duration",
        "trip_category"
    ]
]

fact_taxi_trip.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_date,pickup_hour,PULocationID,DOLocationID,trip_distance,fare_amount,trip_duration,trip_category
0,2025-01-01 00:18:38,2025-01-01 00:26:59,2025-01-01,0,229,237,1.60,10.0,8.350000,Pendek
1,2025-01-01 00:32:40,2025-01-01 00:35:13,2025-01-01,0,236,237,0.50,5.1,2.550000,Pendek
2,2025-01-01 00:44:04,2025-01-01 00:46:01,2025-01-01,0,141,141,0.60,5.1,1.950000,Pendek
3,2025-01-01 00:14:27,2025-01-01 00:20:01,2025-01-01,0,244,244,0.52,7.2,5.566667,Pendek
4,2025-01-01 00:21:34,2025-01-01 00:25:06,2025-01-01,0,244,116,0.66,5.8,3.533333,Pendek


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE fact_taxi_trip AS
SELECT * FROM fact_taxi_trip
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.execute("""
SHOW TABLES
""").fetchdf()

,name
0,dim_location
1,dim_time
2,fact_taxi_trip
3,fact_weather


In [ ]:
query = """
SELECT
    pickup_hour,
    COUNT(*) AS total_trip
FROM fact_taxi_trip
GROUP BY pickup_hour
ORDER BY pickup_hour
"""

con.execute(query).fetchdf()

,pickup_hour,total_trip
0,0,686158
1,1,451034
2,2,295243
3,3,199647
4,4,153601
5,5,163182
6,6,329867
7,7,629843
8,8,862739
9,9,913187


In [ ]:
con.register("dim_time_df", dim_time)
con.register("dim_location_df", dim_location)
con.register("fact_weather_df", fact_weather)
con.register("fact_taxi_trip_df", fact_taxi_trip)

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_time AS
SELECT * FROM dim_time_df
""")

con.execute("""
CREATE OR REPLACE TABLE dim_location AS
SELECT * FROM dim_location_df
""")

con.execute("""
CREATE OR REPLACE TABLE fact_weather AS
SELECT * FROM fact_weather_df
""")

con.execute("""
CREATE OR REPLACE TABLE fact_taxi_trip AS
SELECT * FROM fact_taxi_trip_df
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.execute("""
SHOW TABLES
""").fetchdf()

,name
0,dim_location
1,dim_location_df
2,dim_time
3,dim_time_df
4,fact_taxi_trip
5,fact_taxi_trip_df
6,fact_weather
7,fact_weather_df


In [ ]:
con.close()

In [ ]:
import os

os.listdir(
    "/content/drive/MyDrive/RDV_Project/processed_data"
)

['taxi_raw_ingested.parquet',
 'taxi_cleaned.parquet',
 'taxi_weather.parquet',
 'rdv_project.duckdb']

In [ ]:
import duckdb

db_path = "/content/drive/MyDrive/RDV_Project/processed_data/rdv_project.duckdb"

con = duckdb.connect(db_path)

con.execute("""
SHOW TABLES
""").fetchdf()

,name
0,dim_location
1,dim_time
2,fact_taxi_trip
3,fact_weather
